# MAI verify VietCasualSpeech transcripts

This Kaggle notebook clones the GitHub repo, installs dependencies, loads Kaggle Secrets, then runs the automatic parquet pipeline. Output is written to `/kaggle/working/MAI-verify-transcript/outputs/` so it is preserved when you save/commit the notebook.

## 1. Clone repo

In [ ]:
REPO_URL = "https://github.com/vngclinh/MAI-verify-transcript.git"
WORKDIR = "/kaggle/working/MAI-verify-transcript"

!rm -rf "$WORKDIR"
!git clone "$REPO_URL" "$WORKDIR"

## 2. Install dependencies

In [ ]:
!pip install -q -r "$WORKDIR/requirements.txt"

## 3. Load Kaggle Secrets

Create these Kaggle Secrets before running:

- `HF_TOKEN`
- `AZURE_SPEECH_ENDPOINT`
- `AZURE_SPEECH_KEY`

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
os.environ["AZURE_SPEECH_ENDPOINT"] = secrets.get_secret("AZURE_SPEECH_ENDPOINT")
os.environ["AZURE_SPEECH_KEY"] = secrets.get_secret("AZURE_SPEECH_KEY")

print("Secrets loaded:", all(os.environ.get(k) for k in ["HF_TOKEN", "AZURE_SPEECH_ENDPOINT", "AZURE_SPEECH_KEY"]))

## 4. Optional resume from previous output

If you have a previous output CSV, upload it as a Kaggle Dataset, attach it to this notebook, then set `PREVIOUS_OUTPUT` to that file path. Leave it as `None` for a fresh run.

In [ ]:
from pathlib import Path
import shutil

OUTPUT_DIR = Path(WORKDIR) / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / "viet_casual_speech_code_switching_mai.csv"

# Example: PREVIOUS_OUTPUT = "/kaggle/input/mai-output/viet_casual_speech_code_switching_mai.csv"
PREVIOUS_OUTPUT = None

if PREVIOUS_OUTPUT:
    shutil.copy2(PREVIOUS_OUTPUT, OUTPUT_CSV)
    print("Resuming from", PREVIOUS_OUTPUT)
else:
    print("Fresh run. Output will be", OUTPUT_CSV)

## 5. Run pipeline

This downloads up to 5 pending parquet files, processes them, deletes completed parquet files, then repeats. Use `MAX_PARQUET_BATCHES` for a safety stop during testing; set it to `None` for a full run.

In [ ]:
BATCH_SIZE = 5
SAVE_EVERY = 25
MAX_PARQUET_BATCHES = None  # Example for testing: 1

cmd = [
    "python", "scripts/verify_auto_parquet_batches.py",
    "--output", str(OUTPUT_CSV),
    "--datasets-dir", str(Path(WORKDIR) / "datasets"),
    "--batch-size", str(BATCH_SIZE),
    "--save-every", str(SAVE_EVERY),
]

if MAX_PARQUET_BATCHES is not None:
    cmd += ["--max-parquet-batches", str(MAX_PARQUET_BATCHES)]

%cd $WORKDIR
print("Running:", " ".join(cmd))
import subprocess
subprocess.run(cmd, check=True)

## 6. Inspect output

In [ ]:
import pandas as pd

df = pd.read_csv(OUTPUT_CSV)
print("Output:", OUTPUT_CSV)
print("Rows:", len(df))
if "mai_checked" in df.columns:
    print(df["mai_checked"].astype(str).str.lower().value_counts(dropna=False).head())
df.tail(3)